# 使用 Jupyter AI 创建工业遥测分析平台

本 Notebook 演示如何使用 Jupyter AI（或手动方式）创建一个完整的 FastAPI 项目。

## 项目要求
- 使用 FastAPI + 六边形架构
- 连接 CockroachDB（读写分离）
- Redis 双级缓存（L1 内存 + L2 Redis）

## 如何使用 Jupyter AI

1. 在 JupyterLab 左侧栏点击 AI 聊天图标
2. 在聊天框中输入以下提示词：
   ```
   请帮我创建一个工业遥测分析平台的 FastAPI 项目，使用六边形架构，
   连接 CockroachDB（读写分离）和 Redis 双级缓存。
   ```
3. AI 会生成项目结构和代码
4. 将代码复制到对应文件中

**注意**: 如果 AI 无响应，请检查 Ollama 服务状态。CPU 负载较高时响应可能需要 30-60 秒。
如果持续无响应，可以按以下步骤手动创建。

In [ ]:
# Step 1: 创建项目目录结构
import os

base = '/home/jovyan/work/ai-telemetry-project'
dirs = [
    'src/telemetry/domain',
    'src/telemetry/use_cases',
    'src/telemetry/infrastructure',
    'src/telemetry/api',
    'tests',
    'frontend',
]
for d in dirs:
    os.makedirs(os.path.join(base, d), exist_ok=True)
print('项目目录已创建:', base)
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')

In [ ]:
# Step 2: 创建 pyproject.toml
toml_content = '''[project]
name = "ai-telemetry-project"
version = "1.0.0"
dependencies = ["fastapi", "uvicorn", "sqlalchemy", "redis", "pydantic"]

[tool.pytest.ini_options]
asyncio_mode = "auto"
'''
with open(f'{base}/pyproject.toml', 'w') as f:
    f.write(toml_content)
print('pyproject.toml 已创建')

In [ ]:
# Step 3: 创建领域模型 (domain/models.py)
models_code = '''"""Domain entities for industrial equipment telemetry."""
from dataclasses import dataclass
from datetime import datetime
from typing import Optional


@dataclass(frozen=True)
class TelemetryReading:
    device_id: str
    metric: str
    value: float
    timestamp: datetime
    unit: str = ""

    def __post_init__(self):
        if not self.device_id:
            raise ValueError("device_id is required")
        if self.value != self.value:  # NaN check
            raise ValueError("value cannot be NaN")


@dataclass
class DeviceStatus:
    device_id: str
    status: str  # NORMAL, WARNING, CRITICAL
    anomalies: int = 0
    last_value: float = 0.0
'''
with open(f'{base}/src/telemetry/domain/models.py', 'w') as f:
    f.write(models_code)
print('domain/models.py 已创建')

In [ ]:
# Step 4: 创建阈值分类 (domain/thresholds.py)
thresholds_code = '''"""Domain policy: per-metric warning/critical thresholds."""
DEFAULT_THRESHOLDS = {
    "temperature": (75.0, 95.0),
    "vibration": (5.0, 8.0),
    "pressure": (8.0, 12.0),
}

def classify(metric, value, thresholds=None):
    table = thresholds or DEFAULT_THRESHOLDS
    if metric not in table:
        return "NORMAL"
    warn, crit = table[metric]
    if value >= crit: return "CRITICAL"
    if value >= warn: return "WARNING"
    return "NORMAL"
'''
with open(f'{base}/src/telemetry/domain/thresholds.py', 'w') as f:
    f.write(thresholds_code)
print('domain/thresholds.py 已创建')

In [ ]:
# Step 5: 创建 Redis 双级缓存 (infrastructure/redis_cache.py)
cache_code = '''"""Two-level cache: L1 in-memory LRU + L2 Redis."""
from collections import OrderedDict
import threading

class TwoLevelCache:
    def __init__(self, redis_url=None, max_size=2000):
        self._l1 = OrderedDict()
        self._lock = threading.Lock()
        self._max = max_size
        self._redis = None
        if redis_url:
            try:
                import redis
                self._redis = redis.from_url(redis_url, socket_timeout=0.5)
                self._redis.ping()
            except Exception:
                self._redis = None

    def get(self, key):
        with self._lock:
            if key in self._l1:
                self._l1.move_to_end(key)
                return self._l1[key]
        if self._redis:
            val = self._redis.get(f"telemetry:{key}")
            if val:
                val = val.decode() if isinstance(val, bytes) else val
                self.set(key, val)
                return val
        return None

    def set(self, key, value, ttl=300):
        with self._lock:
            self._l1[key] = value
            self._l1.move_to_end(key)
            while len(self._l1) > self._max:
                self._l1.popitem(last=False)
        if self._redis:
            try:
                self._redis.setex(f"telemetry:{key}", ttl, value)
            except Exception:
                pass

def make_cache():
    import os
    redis_url = os.environ.get("REDIS_URL", "redis://:difyai123456@redis.dify-plus.svc.cluster.local:6379/0")
    return TwoLevelCache(redis_url)
'''
with open(f'{base}/src/telemetry/infrastructure/redis_cache.py', 'w') as f:
    f.write(cache_code)
print('infrastructure/redis_cache.py 已创建')

In [ ]:
# Step 6: 创建 API 路由 (api/routes.py)
routes_code = '''"""FastAPI routes."""
from fastapi import APIRouter, HTTPException, Query
from pydantic import BaseModel, Field
from datetime import datetime
from telemetry.domain.models import TelemetryReading
from telemetry.domain.thresholds import classify
from telemetry.infrastructure.redis_cache import make_cache

router = APIRouter(prefix="/api/v1", tags=["telemetry"])

class TelemetryIn(BaseModel):
    device_id: str = Field(..., min_length=1)
    metric: str = Field(..., min_length=1)
    value: float
    unit: str = ""

class StatusOut(BaseModel):
    device_id: str
    status: str
    last_value: float

@router.post("/telemetry", response_model=StatusOut, status_code=201)
def ingest(body: TelemetryIn):
    st = classify(body.metric, body.value)
    return StatusOut(device_id=body.device_id, status=st, last_value=body.value)

@router.get("/status/{device_id}", response_model=StatusOut)
def status(device_id: str):
    cache = make_cache()
    cached = cache.get(f"status:{device_id}")
    if cached:
        return StatusOut(device_id=device_id, status=cached, last_value=0)
    raise HTTPException(404, "no data")

@router.get("/health")
def health():
    return {"status": "ok", "service": "ai-telemetry"}
'''
with open(f'{base}/src/telemetry/api/routes.py', 'w') as f:
    f.write(routes_code)
print('api/routes.py 已创建')

In [ ]:
# Step 7: 创建 __init__.py 文件
init_files = [
    'src/telemetry/__init__.py',
    'src/telemetry/domain/__init__.py',
    'src/telemetry/use_cases/__init__.py',
    'src/telemetry/infrastructure/__init__.py',
    'src/telemetry/api/__init__.py',
    'tests/__init__.py',
]
for f in init_files:
    path = os.path.join(base, f)
    with open(path, 'w') as fh:
        fh.write('')
print(f'{len(init_files)} 个 __init__.py 已创建')

In [ ]:
# Step 8: 创建测试文件
test_code = '''"""E2E tests for the telemetry API."""
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'src'))
from fastapi.testclient import TestClient
from telemetry.api.routes import router
from fastapi import FastAPI

app = FastAPI()
app.include_router(router)
client = TestClient(app)

def test_health():
    r = client.get("/api/v1/health")
    assert r.status_code == 200
    assert r.json()["status"] == "ok"

def test_ingest_telemetry():
    r = client.post("/api/v1/telemetry", json={"device_id": "PUMP-001", "metric": "temperature", "value": 72.5})
    assert r.status_code == 201
    assert r.json()["status"] == "NORMAL"

def test_ingest_warning():
    r = client.post("/api/v1/telemetry", json={"device_id": "PUMP-001", "metric": "temperature", "value": 80.0})
    assert r.status_code == 201
    assert r.json()["status"] == "WARNING"

def test_ingest_critical():
    r = client.post("/api/v1/telemetry", json={"device_id": "PUMP-001", "metric": "temperature", "value": 110.0})
    assert r.status_code == 201
    assert r.json()["status"] == "CRITICAL"
'''
with open(f'{base}/tests/test_api.py', 'w') as f:
    f.write(test_code)
print('tests/test_api.py 已创建')

In [ ]:
# Step 9: 运行测试验证
import subprocess
result = subprocess.run(
    ['python3', '-m', 'pytest', f'{base}/tests/test_api.py', '-v', '--tb=short'],
    capture_output=True, text=True, cwd=base,
    env={**os.environ, 'PYTHONPATH': f'{base}/src'}
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

## 项目创建完成！

项目结构：
```
ai-telemetry-project/
├── pyproject.toml
├── src/telemetry/
│   ├── domain/
│   │   ├── models.py       # TelemetryReading, DeviceStatus
│   │   └── thresholds.py   # classify() 阈值分类
│   ├── use_cases/          # 业务逻辑层
│   ├── infrastructure/
│   │   └── redis_cache.py  # TwoLevelCache L1+L2
│   └── api/
│       └── routes.py       # FastAPI 路由
├── tests/
│   └── test_api.py         # E2E 测试
└── frontend/
```

## 使用 Jupyter AI 的替代方式

如果 Jupyter AI 可用（Ollama CPU 负载低时），可以直接在聊天中输入：
```
请帮我创建一个工业遥测分析平台的 FastAPI 项目，使用六边形架构，
连接 CockroachDB（读写分离）和 Redis 双级缓存。
```
AI 会生成类似的代码，可以复制到项目中使用。